In [1]:
!nvidia-smi


Sun Feb  9 02:10:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   46C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install flask flask-cors pyngrok diffusers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [4]:
import torch
from torch import autocast
from diffusers import StableDiffusionPipeline
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from PIL import Image
from threading import Thread
import os



app = Flask(__name__)
CORS(app)

# モデルのロード
model_id = "/content/drive/MyDrive/Drebo_training_model_data/400"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to("cuda")

# 画像の保存先ディレクトリ
output_dir = "/content/generated_images"
os.makedirs(output_dir, exist_ok=True)

# 画像グリッド作成関数
def image_grid(imgs, rows, cols):
    assert len(imgs) == rows * cols

    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols * w, rows * h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid

# 静的ファイルを提供するエンドポイント
@app.route('/images/<filename>')
def get_image(filename):
    """保存された画像を公開するエンドポイント"""
    return send_from_directory(output_dir, filename)

# ルートエンドポイント
@app.route("/")
def index():
  return jsonify({"message": "Hello, this is your API server!"})

# 画像生成APIエンドポイント
@app.route('/generate', methods=['POST'])
def generate():
    data = request.json
    prompt = data.get('prompt', '')
    num_images = data.get('num_images', 3)  # デフォルトは6枚
    rows = data.get('rows', 2)  # デフォルトは2行
    cols = data.get('cols', 3)  # デフォルトは3列

    if not prompt:
        return jsonify({'error': 'Prompt is required'}), 400

    # 画像生成処理
    try:
        with autocast("cuda"):
            images = pipe([prompt] * num_images, num_inference_steps=50, guidance_scale=7.5).images

        # グリッド画像の生成
        grid = image_grid(images, rows=rows, cols=cols)
        output_path = os.path.join(output_dir, "generated_grid.png")
        grid.save(output_path)

        # 画像urlを返す
        return jsonify({'image_url': f'/images/{os.path.basename(output_path)}'})  # 保存先パスを返却

    except Exception as e:
        return jsonify({'error': str(e)}), 500

# スレッドでサーバーをバックグラウンド実行
def run():
  app.run(host='0.0.0.0', port=5000)

if __name__ == '__main__':
  thread = Thread(target=run)
  thread.start()


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


# サーバーシャットダウン機能つき

なんか画像表示されなかった
とりあえず正常に動かしたかったら上のセル実行して

# ngrokサーバー

In [5]:
!pip install pyngrok
from pyngrok import ngrok

ngrok.set_auth_token("2rn06XHBOtxvwEIeJi07FtCrv5Y_t2LGPMPvwY8kAPJmh5bm")

public_url = ngrok.connect(5000)  # Flaskサーバーのポートに接続
print(f'Public URL: {public_url}')


 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Public URL: NgrokTunnel: "https://2b00-34-125-202-132.ngrok-free.app" -> "http://localhost:5000"


# サーバーシャットダウン

# サーバーが動いてるか確認するやつ

なんかここに表示されてるやつはサーバー動いてるかどうかじゃなくてflaskコマンドとかシェルが実行されたことを示すものらしい

In [8]:
!ps aux | grep flask


root       13205  0.0  0.0   7376  3472 ?        S    02:49   0:00 /bin/bash -c ps aux | grep flask
root       13207  0.0  0.0   6484  2276 ?        S    02:49   0:00 grep flask


In [7]:
!ps aux | grep ngrok


root        4254 26.0  0.0 1275528 43084 ?       Sl   02:14   0:00 /root/.config/ngrok/ngrok start -
root        4281  0.0  0.0   7376  3488 ?        S    02:14   0:00 /bin/bash -c ps aux | grep ngrok
root        4283  0.0  0.0   6484  2432 ?        S    02:14   0:00 grep ngrok
